# connections-rl — 7B scale-ablation eval on Kaggle 2×T4

Same protocol as the 1.5B eval (same 162-puzzle held-out test split), swapping
in Qwen2.5-7B-Instruct + the 7B adapters. 7B fp16 (~15GB) needs BOTH T4s:
vLLM runs with tensor parallelism (`-tp 2`) and explicit fp16 (T4 has no bf16).

Settings → Accelerator → **GPU T4 x2**, Internet → On. Interactive is fine.
When cell 3 finishes, **download `/kaggle/working/results-7b.zip` immediately**.

In [ ]:
# Cell 1 — setup: repo, data, 7B adapters from the Hub
import os
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
HF_USER = 'jacksonlukas'

!git clone https://github.com/jacksonmlukas/connections-rl.git
%cd connections-rl
!pip install -q -e . openai vllm
!git clone --depth 1 https://github.com/jacksonmlukas/gvc-local.git /kaggle/working/gvc-local
os.environ['CONNECTIONS_PUZZLES'] = '/kaggle/working/gvc-local/data/puzzles/tagged_connections.json'
!python -m connections_rl.data.build --out data/splits

from huggingface_hub import snapshot_download
for name in ['sft-7b', 'grpo-7b']:
    snapshot_download(f'{HF_USER}/connections-rl-{name}',
                      local_dir=f'adapters/{name}', token=os.environ['HF_TOKEN'])
!ls adapters/sft-7b adapters/grpo-7b

In [ ]:
# Cell 2 — vLLM: 7B sharded across both T4s (tp=2), fp16, both LoRAs mounted
import subprocess, time, urllib.request

A = '/kaggle/working/connections-rl/adapters'
# --enforce-eager: CUDA-graph capture OOMs on 14.5GB T4s with 7B+LoRA;
# eager decoding is slower per token but trivial for ~500 short completions.
proc = subprocess.Popen(
    f'vllm serve Qwen/Qwen2.5-7B-Instruct --dtype half --tensor-parallel-size 2 '
    f'--enable-lora --enforce-eager '
    f'--lora-modules connections-rl-sft-7b={A}/sft-7b connections-rl-grpo-7b={A}/grpo-7b '
    f'--max-model-len 2048 --gpu-memory-utilization 0.85',
    shell=True,
    stdout=open('/kaggle/working/vllm.log', 'w'), stderr=subprocess.STDOUT)

for _ in range(150):  # 7B download + tp startup can take ~10-15 min
    try:
        urllib.request.urlopen('http://localhost:8000/health')
        print('vLLM ready')
        break
    except Exception:
        time.sleep(10)
else:
    raise RuntimeError('vLLM failed — check /kaggle/working/vllm.log')

In [ ]:
# Cell 3 — full 7B eval (base / sft / grpo, paired stats) + bundle results
!python -m connections_rl.eval.run --config configs/eval/qwen7b.yaml
!zip -qr /kaggle/working/results-7b.zip results-7b
print('done — download /kaggle/working/results-7b.zip from the output panel')